# Visium HD Artifact Cleanup

Demonstration notebook showing how to load a Visium HD sample with `Artifact_remove_HD`,
remove artifacts, and save the updated outputs.


In [ ]:
import os, sys
from pathlib import Path

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from artifactsRemoval import Artifact_remove_HD

data_dir = '/path/to/Visium_HD/sample'  # <- update this path
positions_path = os.path.join(data_dir, 'spatial', 'tissue_positions.parquet')
h5_path = os.path.join(data_dir, 'filtered_feature_cell_matrix.h5')
print('Using data_dir:', data_dir)
print('Positions exists?', os.path.exists(positions_path))
print('HDF5 exists?', os.path.exists(h5_path))


In [ ]:
hd = Artifact_remove_HD(
    data_dir=data_dir,
    positions_path=positions_path,
    h5_path=h5_path,
    neighbor_mode='k',
    k_neighbors=8,
)

print('Barcodes:', hd.barcode_list.shape[0])
print('Features:', hd.feature_list.shape[0])
print('Positions:', hd.tissue_position.shape[0])


In [ ]:
# Remove artifacts
hd.remove_border(distance=float(hd._grid_radius) if hd._grid_radius else 0.0)
hd.remove_edge(distance=2)
hd.remove_malfunction()  # clustered removal (min cluster size 15)


In [ ]:
# Save outputs
output_dir = os.path.join(data_dir, 'blade_outputs')
hd.save(output_dir)
out_h5 = os.path.join(output_dir, 'filtered_feature_cell_matrix.filtered.h5')
hd.save_hdf5(out_h5)
